In [1]:
# === DATASET PROFILING ===
# Run this BEFORE the benchmark to document dataset context

import pandas as pd
import json

# Load raw DataFrames (not PandasAI wrapped)
tables = {
    "cc_clinic_level": pd.read_csv("cc_clinic_level.csv"),
    "cc_doctor": pd.read_csv("cc_doctor.csv"),
    "cc_hourly": pd.read_csv("cc_hourly.csv"),
    "cc_patient": pd.read_csv("cc_patient.csv"),
}

print("=" * 80)
print("📋 DATASET PROFILE")
print("=" * 80)

# --- Domain Justification ---
print("\n🏥 Domain: Healthcare Clinic Management")
print("   Justification: Healthcare data involves multi-table relationships,")
print("   mixed data types (categorical, numerical, temporal), and real-world")
print("   analytical queries (revenue analysis, patient demographics, doctor")
print("   performance) — making it a strong test for Text-to-SQL generation.")

# --- Data Origin ---
# ⚠️ UPDATE THIS based on your actual data source
DATA_ORIGIN = "Synthetic"  # Change to "Real (anonymized)" if applicable
print(f"\n📦 Data Origin: {DATA_ORIGIN}")
if DATA_ORIGIN == "Synthetic":
    print("   Note: Synthetic data generated to simulate a multi-clinic healthcare system.")
else:
    print("   Note: Real-world data, anonymized for privacy compliance.")

# --- Table-level Stats ---
print("\n📊 Table-Level Statistics")
print("-" * 80)
print(f"{'Table':<25} {'Rows':>8} {'Columns':>8} {'Numeric Cols':>12} {'Categorical':>12} {'Missing %':>10}")
print("-" * 80)

total_rows = 0
total_cols = 0
table_profiles = {}

for name, df in tables.items():
    rows, cols = df.shape
    total_rows += rows
    total_cols += cols
    numeric_cols = len(df.select_dtypes(include='number').columns)
    categorical_cols = len(df.select_dtypes(include='object').columns)
    missing_pct = (df.isnull().sum().sum() / (rows * cols)) * 100
    
    table_profiles[name] = {
        "rows": rows,
        "columns": cols,
        "numeric_columns": numeric_cols,
        "categorical_columns": categorical_cols,
        "missing_pct": round(missing_pct, 2),
        "column_names": list(df.columns),
        "dtypes": {col: str(dtype) for col, dtype in df.dtypes.items()},
    }
    
    print(f"{name:<25} {rows:>8,} {cols:>8} {numeric_cols:>12} {categorical_cols:>12} {missing_pct:>9.1f}%")

print("-" * 80)
print(f"{'TOTAL':<25} {total_rows:>8,} {total_cols:>8}")

# --- Schema Complexity ---
print("\n\n🔗 Schema Complexity Analysis")
print("-" * 80)

# Detect potential join keys (shared column names across tables)
all_columns = {name: set(df.columns) for name, df in tables.items()}
join_keys = {}
for i, (name1, cols1) in enumerate(all_columns.items()):
    for name2, cols2 in list(all_columns.items())[i+1:]:
        shared = cols1 & cols2
        if shared:
            join_keys[f"{name1} ↔ {name2}"] = list(shared)

print("Potential Join Keys (shared columns):")
if join_keys:
    for pair, keys in join_keys.items():
        print(f"  • {pair}: {keys}")
else:
    print("  • No shared column names detected (joins may use implicit relationships)")

num_tables = len(tables)
num_join_pairs = len(join_keys)
avg_join_depth = num_join_pairs / max(num_tables - 1, 1)

# Schema complexity scoring
complexity_score = 0
complexity_score += min(num_tables, 5)           # Max 5 pts for tables
complexity_score += min(total_cols // 10, 5)      # Max 5 pts for columns
complexity_score += min(num_join_pairs * 2, 5)    # Max 5 pts for joins
complexity_score += min(total_rows // 1000, 5)    # Max 5 pts for volume

if complexity_score >= 15:
    complexity_level = "High"
elif complexity_score >= 8:
    complexity_level = "Medium"
else:
    complexity_level = "Low"

print(f"\n📐 Schema Complexity Metrics:")
print(f"  • Number of tables:      {num_tables}")
print(f"  • Total columns:         {total_cols}")
print(f"  • Total records:         {total_rows:,}")
print(f"  • Join pairs detected:   {num_join_pairs}")
print(f"  • Average join depth:    {avg_join_depth:.1f}")
print(f"  • Complexity score:      {complexity_score}/20")
print(f"  • Complexity level:      {complexity_level}")

# --- Column Details ---
print("\n\n📝 Detailed Column Schema")
print("-" * 80)
for name, df in tables.items():
    print(f"\n  📁 {name} ({df.shape[0]} rows × {df.shape[1]} cols)")
    for col in df.columns:
        dtype = df[col].dtype
        nunique = df[col].nunique()
        sample = str(df[col].dropna().iloc[0])[:30] if len(df[col].dropna()) > 0 else "N/A"
        print(f"      • {col:<30} {str(dtype):<10} unique={nunique:<6} sample: {sample}")

# --- Question Complexity Mapping ---
print("\n\n❓ Question Complexity Breakdown")
print("-" * 80)
question_complexity = {
    "Easy (5 questions)": {
        "operations": ["SELECT", "MAX/MIN", "AVG", "DISTINCT", "COUNT"],
        "tables_needed": "1 (single table)",
        "join_depth": 0,
        "description": "Single-table queries with basic aggregation"
    },
    "Medium (5 questions)": {
        "operations": ["GROUP BY", "SUM", "AVG with filter", "Conditional aggregation"],
        "tables_needed": "1-3 (may need joins)",
        "join_depth": 1,
        "description": "Multi-table queries with grouping and filtering"
    },
    "Hard (5 questions)": {
        "operations": ["Derived columns", "Ratio calculations", "Correlation", "Subqueries", "Multi-condition filters"],
        "tables_needed": "2-4 (complex joins)",
        "join_depth": 2,
        "description": "Complex analytical queries with derived metrics and multi-table joins"
    }
}

for level, details in question_complexity.items():
    print(f"\n  {level}")
    print(f"    Tables needed:  {details['tables_needed']}")
    print(f"    Join depth:     {details['join_depth']}")
    print(f"    SQL operations: {', '.join(details['operations'])}")
    print(f"    Description:    {details['description']}")

# --- Save profile as JSON for reference ---
profile = {
    "domain": "Healthcare Clinic Management",
    "data_origin": DATA_ORIGIN,
    "total_tables": num_tables,
    "total_records": total_rows,
    "total_columns": total_cols,
    "join_pairs": num_join_pairs,
    "avg_join_depth": avg_join_depth,
    "complexity_score": complexity_score,
    "complexity_level": complexity_level,
    "tables": table_profiles,
    "join_keys": join_keys,
}

with open("exports/dataset_profile.json", "w") as f:
    json.dump(profile, f, indent=2)
print(f"\n💾 Dataset profile saved to: exports/dataset_profile.json")

# --- Summary Box for FYP Report ---
print("\n" + "=" * 80)
print("📋 COPY THIS TO YOUR FYP REPORT")
print("=" * 80)
print(f"""
Dataset Profile Summary
─────────────────────────────────────────
Domain:              Healthcare Clinic Management
Data Origin:         {DATA_ORIGIN}
Number of Tables:    {num_tables}
Total Records:       {total_rows:,}
Total Columns:       {total_cols}
Join Pairs:          {num_join_pairs}
Average Join Depth:  {avg_join_depth:.1f}
Schema Complexity:   {complexity_level} ({complexity_score}/20)
Question Categories: Easy (5), Medium (5), Hard (5) = 15 total

─────────────────────────────────────────
""")

📋 DATASET PROFILE

🏥 Domain: Healthcare Clinic Management
   Justification: Healthcare data involves multi-table relationships,
   mixed data types (categorical, numerical, temporal), and real-world
   analytical queries (revenue analysis, patient demographics, doctor
   performance) — making it a strong test for Text-to-SQL generation.

📦 Data Origin: Synthetic
   Note: Synthetic data generated to simulate a multi-clinic healthcare system.

📊 Table-Level Statistics
--------------------------------------------------------------------------------
Table                         Rows  Columns Numeric Cols  Categorical  Missing %
--------------------------------------------------------------------------------
cc_clinic_level                 46       20           16            4       0.0%
cc_doctor                      231       17           11            6       0.0%
cc_hourly                  157,419       21           16            5      11.4%
cc_patient                 241,159       14